# Dense retrieval + BM25 fusion (RRF)

Этот notebook принимает тяжёлые GPU-ранги из Kaggle и отвечает на три вопроса:

1. Насколько русский `deepvk/USER-bge-m3` хорош сам по себе на полном корпусе?
2. Находит ли dense релевантные items, которые пропускает лексический BM25?
3. Улучшает ли Reciprocal Rank Fusion Recall@50 на невидимой половине holdout?

Порядок внутри top-50 в соревновании не важен, но RRF нужен, чтобы выбрать ровно 50 items из объединения каналов без калибровки несовместимых BM25/cosine scores.

In [1]:
from pathlib import Path
import json, re, sys
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src'))
from avito_retrieval.bm25 import SparseBM25
from avito_retrieval.channels import filtered_ranking_from_scores, top_rows
from avito_retrieval.fusion import reciprocal_rank_fusion
from avito_retrieval.metrics import recall_at_k, hit_rate_at_k
from avito_retrieval.text import query_text

DENSE_DIR = ROOT/'artifacts/dense_kaggle'
RANK_DIR = ROOT/'artifacts/rankings'
RANK_DIR.mkdir(parents=True, exist_ok=True)
from dataclasses import asdict
from avito_retrieval.statistics import paired_recall_test

## 1. Целостность GPU-артефактов

Kaggle должен вернуть по одной строке для каждого validation и benchmark query. Каждый dense список уже агрегирован passage → unique item и прошёл category/rating filter. Здесь повторно проверяем query coverage, допустимость `item_id` и отсутствие дублей: высокая offline-метрика бесполезна, если итоговый CSV нарушает контракт.

In [2]:
manifest = pd.read_parquet(ROOT/'artifacts/validation/manifest.parquet')
labels = pd.read_parquet(ROOT/'artifacts/validation/labels.parquet')
relevant = labels.groupby('eval_query_id').item_id.agg(set).to_dict()
benchmark = pd.read_parquet(ROOT/'dataset/benchmark_queries.parquet')
items = pd.read_parquet(ROOT/'artifacts/bm25/items.parquet')
valid_items = set(items.item_id.astype(str))
dense = pd.read_parquet(DENSE_DIR/'dense_rankings.parquet')
run_info = json.loads((DENSE_DIR/'dense_run.json').read_text())
display(run_info)
assert len(dense)==len(manifest)+len(benchmark)
assert set(dense[dense.split.eq('validation')].query_key)==set(manifest.eval_query_id)
assert set(dense[dense.split.eq('benchmark')].query_key)==set(benchmark.query_id)

def parse_rank(value):
    return [] if not value else str(value).split()
rank_columns=['dense_global','dense_local','user_first_global','user_first_local','e5_first_global','e5_first_local']
assert set(rank_columns)<=set(dense.columns)
for column in rank_columns:
    for rank in dense[column].map(parse_rank):
        assert len(rank)==len(set(rank)) and set(rank)<=valid_items
print('Dense artefacts are valid:', dense.shape)

{'models': ['deepvk/USER-bge-m3', 'intfloat/multilingual-e5-small'],
 'items': 189212,
 'passages': 554920,
 'queries': 4904,
 'chunk_words': 140,
 'overlap': 30,
 'max_chunks': 4,
 'gpus': ['Tesla T4', 'Tesla T4']}

Dense artefacts are valid: (4904, 8)


## 2. Dev/test внутри фиксированного manifest

RRF имеет небольшие гиперпараметры (`rrf_k` и вес dense). Чтобы не выбрать их на тех же запросах, по которым отчитываемся, manifest делится 50/50 с сохранением primary strata `has_filters × local_pool_bucket`. Grid выбирается только на dev; test используется один раз для итогового сравнения. После выбора веса финальный `answer.csv` строится на всех train labels и benchmark без скрытой разметки.

In [3]:
manifest = manifest.copy(); manifest['fold']=-1
splitter = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
for fold, (_, indices) in enumerate(splitter.split(manifest, manifest.primary_stratum)):
    manifest.loc[indices,'fold']=fold
display(pd.crosstab(manifest.primary_stratum, manifest.fold, normalize='index').round(3))

fold,0,1
primary_stratum,,
False|0,0.500,0.500
False|1-99,0.517,0.483
False|100-499,0.500,0.500
False|10000+,0.499,0.501
False|2000-9999,0.500,0.500
False|500-1999,0.500,0.500
True|0,0.503,0.497
True|1-99,0.462,0.538
True|100-499,0.506,0.494


## 3. BM25 rankings для тех же validation и benchmark queries

Индекс и tokenizer не перестраиваются. Для каждого запроса одним sparse matrix multiplication считаются scores, затем формируются global и same-location каналы с теми же category/rating rules. Сохраняем top-250: RRF должен выбирать top-50 из более широкого пула, иначе поздний релевантный item не сможет подняться за счёт второго retriever.

In [4]:
bm25 = SparseBM25.load(ROOT/'artifacts/bm25')
validation_queries = manifest.copy(); validation_queries['query_key']=validation_queries.eval_query_id; validation_queries['split']='validation'
benchmark_queries = benchmark.copy(); benchmark_queries['query_key']=benchmark_queries.query_id; benchmark_queries['split']='benchmark'
query_rows = pd.concat([validation_queries, benchmark_queries], ignore_index=True)
bm25_records=[]
for start in range(0,len(query_rows),64):
    batch=query_rows.iloc[start:start+64]
    scores=bm25.score_many([query_text(row) for _,row in batch.iterrows()])
    for col,(_,query) in enumerate(batch.iterrows()):
        rank=filtered_ranking_from_scores(query,items,scores[:,col],retrieve_k=300,output_k=250)
        plain_rows=top_rows(scores[:,col],np.ones(len(items),dtype=bool),250)
        bm25_records.append({'query_key':query.query_key,'split':query.split,'bm25':' '.join(rank),'bm25_plain':' '.join(items.iloc[plain_rows].item_id)})
    print(f'{min(start+64,len(query_rows))}/{len(query_rows)}',end='\r')
bm25_ranks=pd.DataFrame(bm25_records)
bm25_ranks.to_parquet(RANK_DIR/'bm25_rankings.parquet',index=False)

## 4. Dense-only, model/chunk ablation и комплементарность

Три варианта образуют контролируемый эксперимент: USER с 4 chunks, USER с first chunk и E5 с тем же first chunk. Выбор dense-варианта делается только по dev Recall@50; на test затем считаются парные CI и randomization p-values для `4 chunks − 1 chunk` и `USER − E5`.

Для каждого варианта global и local ranks объединяются одинаковым RRF. Низкий BM25/dense overlap при ненулевом dense recall означает полезную комплементарность; высокий overlap при более низком recall — повторение BM25 с добавлением шума.

In [5]:
joined=dense.merge(bm25_ranks,on=['query_key','split'],validate='one_to_one')
validation=joined[joined.split.eq('validation')].set_index('query_key')
bm25_pred={qid:parse_rank(row.bm25) for qid,row in validation.iterrows()}

def dense_variant(prefix):
    return {qid:reciprocal_rank_fusion(
        [parse_rank(row[f'{prefix}_global']),parse_rank(row[f'{prefix}_local'])],
        weights=[1,1.15],top_k=250,
    ) for qid,row in validation.iterrows()}

dense_variants={
    'USER_4_chunks': dense_variant('dense'),
    'USER_1_chunk': dense_variant('user_first'),
    'E5_1_chunk': dense_variant('e5_first'),
}
dense_global={qid:parse_rank(row.dense_global) for qid,row in validation.iterrows()}
fold_by_id=manifest.set_index('eval_query_id').fold.to_dict()
dev_ids={q for q,f in fold_by_id.items() if f==0}; test_ids=set(fold_by_id)-dev_ids
def subset(mapping,ids): return {q:mapping[q] for q in ids}
variant_scores=pd.DataFrame([
    {'method':name,
     'dev_recall':recall_at_k(subset(pred,dev_ids),subset(relevant,dev_ids)),
     'test_recall':recall_at_k(subset(pred,test_ids),subset(relevant,test_ids)),
     'full_recall':recall_at_k(pred,relevant)}
    for name,pred in dense_variants.items()
]).sort_values('dev_recall',ascending=False)
selected_dense_name=str(variant_scores.iloc[0].method)
dense_pred=dense_variants[selected_dense_name]
display(variant_scores)
overlap=np.mean([len(set(bm25_pred[q][:50])&set(dense_pred[q][:50]))/50 for q in bm25_pred])
print('Selected on dev:',selected_dense_name)
print(f'Mean BM25/dense top-50 overlap: {overlap:.3f}')

,method,dev_recall,test_recall,full_recall
0,USER_4_chunks,0.773491,0.762806,0.768148
2,E5_1_chunk,0.751604,0.726427,0.739016
1,USER_1_chunk,0.748341,0.725544,0.736943


Selected on dev: USER_4_chunks
Mean BM25/dense top-50 overlap: 0.335


## 5. RRF grid: настройка на dev, один отчёт на test

RRF начисляет item балл `weight / (rrf_k + rank)`. Малый `rrf_k` сильнее доверяет первым позициям; большой делает вклад хвоста более равномерным. Scores моделей не смешиваются напрямую, поэтому различный масштаб BM25 и cosine не влияет на fusion. Grid маленький и заранее фиксированный.

In [6]:
grid=[]; hybrid_cache={}
for rrf_k in [20,60,100]:
  for dense_weight in [.25,.5,.75,1.,1.25,1.5,2.]:
    pred={q:reciprocal_rank_fusion([bm25_pred[q],dense_pred[q]],weights=[1,dense_weight],rrf_k=rrf_k,top_k=50) for q in relevant}
    hybrid_cache[(rrf_k,dense_weight)]=pred
    grid.append({'rrf_k':rrf_k,'dense_weight':dense_weight,'dev_recall':recall_at_k(subset(pred,dev_ids),subset(relevant,dev_ids))})
grid=pd.DataFrame(grid).sort_values(['dev_recall','dense_weight'],ascending=[False,True])
display(grid.head(10))
best=grid.iloc[0]; best_key=(int(best.rrf_k),float(best.dense_weight)); hybrid_pred=hybrid_cache[best_key]
comparison=pd.DataFrame([
 {'method':'BM25','dev':recall_at_k(subset(bm25_pred,dev_ids),subset(relevant,dev_ids)),'test':recall_at_k(subset(bm25_pred,test_ids),subset(relevant,test_ids)),'full':recall_at_k(bm25_pred,relevant)},
 {'method':selected_dense_name,'dev':recall_at_k(subset(dense_pred,dev_ids),subset(relevant,dev_ids)),'test':recall_at_k(subset(dense_pred,test_ids),subset(relevant,test_ids)),'full':recall_at_k(dense_pred,relevant)},
 {'method':f'RRF {best_key}','dev':recall_at_k(subset(hybrid_pred,dev_ids),subset(relevant,dev_ids)),'test':recall_at_k(subset(hybrid_pred,test_ids),subset(relevant,test_ids)),'full':recall_at_k(hybrid_pred,relevant)},
])
display(comparison)
stat_tests={
 'USER_4_vs_1_chunk_test':asdict(paired_recall_test(dense_variants['USER_4_chunks'],dense_variants['USER_1_chunk'],relevant,query_ids=sorted(test_ids))),
 'USER_vs_E5_first_chunk_test':asdict(paired_recall_test(dense_variants['USER_1_chunk'],dense_variants['E5_1_chunk'],relevant,query_ids=sorted(test_ids))),
 'RRF_vs_BM25_test':asdict(paired_recall_test(hybrid_pred,bm25_pred,relevant,query_ids=sorted(test_ids))),
 'dense_local_RRF_vs_global_test':asdict(paired_recall_test(dense_variants['USER_4_chunks'],dense_global,relevant,query_ids=sorted(test_ids))),
}
display(pd.DataFrame(stat_tests).T)

,rrf_k,dense_weight,dev_recall
4,20,1.25,0.831430
11,60,1.25,0.829935
12,60,1.50,0.829527
5,20,1.50,0.828575
3,20,1.00,0.828032
6,20,2.00,0.827488
2,20,0.75,0.827352
10,60,1.00,0.826944
19,100,1.50,0.825041
17,100,1.00,0.823681


,method,dev,test,full
0,BM25,0.752923,0.747866,0.750394
1,USER_4_chunks,0.773491,0.762806,0.768148
2,"RRF (20, 1.25)",0.831430,0.809203,0.820317


,mean_delta,ci_low,ci_high,p_value_greater,n_queries
USER_4_vs_1_chunk_test,0.037262,0.022240,0.052298,0.000050,1226.0
USER_vs_E5_first_chunk_test,-0.000884,-0.018760,0.017129,0.536773,1226.0
RRF_vs_BM25_test,0.061338,0.044113,0.079078,0.000050,1226.0
dense_local_RRF_vs_global_test,0.462167,0.434122,0.489927,0.000050,1226.0


## 6. Slice metrics и решение о SPLADE/ColBERT

Aggregate Recall может скрыть регрессию на tail-запросах или больших локациях. Поэтому считаем BM25/RRF на каждом фильтровом и corpus-size slice. Инженерное изменение принимается как подтверждённое, когда test-fold имеет положительный effect, 95% paired bootstrap CI выше нуля и one-sided randomization `p < 0.05`; p-value без размера эффекта не используется.

Для решения о следующих тяжёлых экспериментах заранее задаём практический критерий: продолжаем со SPLADE и затем ColBERT, если test Recall@50 гибрида ниже 0.85 **и** oracle union BM25+dense оставляет не менее 2 п.п. потенциального recall. Эксперименты запускаются только **если позволяют доступные вычислительные ресурсы**: GPU-время, VRAM и место для индексов. Это не гарантирует прирост, но показывает измеримый запас, ради которого оправданы дополнительные затраты. SPLADE проверяет learned sparse retrieval, а ColBERT — late interaction токенов; каждый подход оформляется отдельным notebook и принимается только по парному тесту на том же holdout.

In [7]:
slice_rows=[]
for column in ['has_filters','local_pool_bucket','query_frequency_bucket','positive_bucket']:
  for value,group in manifest.groupby(column):
    ids=set(group.eval_query_id)
    slice_rows.append({'slice':column,'value':str(value),'queries':len(ids),'bm25':recall_at_k(subset(bm25_pred,ids),subset(relevant,ids)),'rrf':recall_at_k(subset(hybrid_pred,ids),subset(relevant,ids))})
slices=pd.DataFrame(slice_rows); slices['delta']=slices.rrf-slices.bm25
display(slices.sort_values(['slice','value']))
rrf_test=float(comparison.iloc[-1]['test']); bm25_test=float(comparison.iloc[0]['test'])
test_gain=rrf_test-bm25_test
union_pred={q:list(dict.fromkeys([*bm25_pred[q],*dense_pred[q]])) for q in test_ids}
oracle_union_recall=recall_at_k(union_pred,subset(relevant,test_ids),k=500)
oracle_gap=oracle_union_recall-rrf_test
advanced_retrieval_recommended = bool(rrf_test < .85 and oracle_gap >= .02)
print(f'Unseen-test RRF gain: {test_gain:+.4f}')
print(f'Candidate-union recall upper bound: {oracle_union_recall:.4f}; gap={oracle_gap:.4f}')
print('Try SPLADE, then ColBERT if compute resources allow:',advanced_retrieval_recommended)

,slice,value,queries,bm25,rrf,delta
0,has_filters,False,1546,0.754991,0.833495,7.850367e-02
1,has_filters,True,906,0.742550,0.797829,5.527962e-02
2,local_pool_bucket,0,427,0.419594,0.483997,6.440281e-02
3,local_pool_bucket,1-99,42,0.833333,0.857143,2.380952e-02
4,local_pool_bucket,100-499,213,0.833333,0.877934,4.460094e-02
5,local_pool_bucket,10000+,543,0.741191,0.803867,6.267649e-02
6,local_pool_bucket,2000-9999,554,0.838448,0.921029,8.258123e-02
7,local_pool_bucket,500-1999,673,0.863794,0.943536,7.974245e-02
12,positive_bucket,1,2283,0.749452,0.820412,7.095926e-02
13,positive_bucket,2,128,0.769531,0.828125,5.859375e-02


Unseen-test RRF gain: +0.0613
Candidate-union recall upper bound: 0.9409; gap=0.1317
Try SPLADE, then ColBERT if compute resources allow: True


## 7. Benchmark prediction и строгая проверка CSV

После выбора гиперпараметров на dev тот же deterministic fusion применяется к benchmark. Проверяем ровно две колонки, все query IDs, максимум 50 уникальных items, формат hex и принадлежность корпусу. Это ещё research output; после подтверждения лучшей схемы логика переносится в отдельный batch inference pipeline.

In [8]:
bench=joined[joined.split.eq('benchmark')].set_index('query_key')
prefix_by_variant={'USER_4_chunks':'dense','USER_1_chunk':'user_first','E5_1_chunk':'e5_first'}
selected_prefix=prefix_by_variant[selected_dense_name]
answers=[]
for query_id,row in bench.iterrows():
    dense_rank=reciprocal_rank_fusion([parse_rank(row[f'{selected_prefix}_global']),parse_rank(row[f'{selected_prefix}_local'])],weights=[1,1.15],top_k=250)
    final=reciprocal_rank_fusion([parse_rank(row.bm25),dense_rank],weights=[1,best_key[1]],rrf_k=best_key[0],top_k=50)
    answers.append({'query_id':query_id,'answer':' '.join(final)})
answer=pd.DataFrame(answers)
assert list(answer.columns)==['query_id','answer'] and len(answer)==len(benchmark)
assert answer.query_id.is_unique and set(answer.query_id)==set(benchmark.query_id)
for value in answer.answer:
    ids=value.split(); assert 0<len(ids)<=50 and len(ids)==len(set(ids))
    assert set(ids)<=valid_items and all(re.fullmatch(r'[0-9a-f]{16}',x) for x in ids)
answer.to_csv(ROOT/'answer.csv',index=False,encoding='utf-8')
metrics={'models':run_info['models'],'selected_dense':selected_dense_name,'rrf_k':best_key[0],'dense_weight':best_key[1],'test_gain':test_gain,'oracle_union_recall':oracle_union_recall,'comparison':comparison.to_dict('records'),'paired_tests':stat_tests,'advanced_retrieval_recommended':advanced_retrieval_recommended,'advanced_retrieval_requires_available_compute':True}
(ROOT/'reports/dense_rrf_metrics.json').write_text(json.dumps(metrics,ensure_ascii=False,indent=2),encoding='utf-8')
display(answer.head()); print('answer.csv validated:',answer.shape)

,query_id,answer
0,70DfDUpwjxB4lzFd,355392014208b7bf 6042fecdd7753112 603623b4bd9e...
1,JTrdTaZJvSiLPkXj,cbeccbecb1fb8d86 8703b2a0442c5d25 16ea26ff0aa0...
2,LZCZNoVG4AFUkVRJ,dab52187b4500d9b 3b370cc603f67947 d8fce513e4f0...
3,660ac9QVtXkRxZC3,af91ec4a29b66636 a846a2a5241e1180 bd2cb4500fad...
4,YgHcM9MVbxKnxD1e,1a62634394544781 13f58fe9542bdb1a 615c73ea4c3b...


answer.csv validated: (2452, 2)
